## Active user detection with randomly pivoted Cholesky (RPC) decomposition

In [1]:
# Author: Saikat Majumder
# Date: October, 2026
# Place: Raipur, India
# RPC libraries taken from https://github.com/eepperly/Randomly-Pivoted-Cholesky

In [2]:
import time
import numpy as np
import scipy.io as sio
from sklearn.metrics import mean_absolute_error as mae
from sklearn.base import ClassifierMixin, BaseEstimator
from sklearn.metrics.pairwise import pairwise_kernels
from rpcholesky import rpcholesky
from matrix import PSDMatrix
import tracemalloc

In [3]:
mat_contents = sio.loadmat('aud_dataset.mat') 
Xtra = mat_contents['train_data']
Ytra = mat_contents['train_target']
Xtst = mat_contents['test_data']
Ytst = mat_contents['test_target']

In [4]:
class KELMRPC(ClassifierMixin, BaseEstimator):
    def __init__(self, C=1, method="full", num_pivots=1000, S=None, eps=1e-5, kernel="linear", gamma=None, degree=3, coef0=1, kernel_params=None):
        self.C = C
        self.kernel = kernel
        self.method = method
        self.S = S
        self.eps = eps
        self.gamma = 0.01
        self.degree = degree
        self.coef0 = coef0
        self.num_pivots = num_pivots
        self.kernel_params = kernel_params

    def fit(self, X, Y):
        """
        Fit the model from the data in X and the labels in y.
        Parameters
        ----------
        X : array-like, shape (N x d)
            Training vector, where N is the number of samples, and d is the number of features.
        y : array-like, shape (N)
            Labels vector, where N is the number of samples.
        Returns
        -------
        self : object
            Returns the instance itself.
        """
        self.X, self.Y = X, Y
        n, d = X.shape
        T = self.Y
        K = self._get_kernel(X)

        if self.method == "full":
            self.beta = np.linalg.inv((1 / self.C) * np.eye(n) + K).dot(T)
        else:
            num_pivots = self.num_pivots
            F = rpcholesky(PSDMatrix(K), num_pivots).get_left_factor() # Nystrom approximation is F @ F.T
            z = self.C*T
            u = (F.T).dot(T)
            s = np.linalg.inv(np.eye(num_pivots) + self.C*np.dot(F.T,F)).dot(u)
            self.beta = z - (self.C*self.C)*np.dot(F,s)
        return self

    def predict(self, X):
        """
        Predict the class labels for the provided data.
        Parameters
        ----------
        X : array-like, shape (N x d)
            Test samples. N is the number of samples and d the number of features.
        Returns
        -------
        y : array of shape (N)
            Class labels for each data sample.
        """
        K = self._get_kernel(X, self.X)
        predictions = K.dot(self.beta)
        return predictions

    def _get_kernel(self, X, Y=None):
        if callable(self.kernel):
            params = self.kernel_params or {}
        else:
            params = {'gamma': self.gamma,
                      'degree': self.degree,
                      'coef0': self.coef0}
        return pairwise_kernels(X, Y, metric=self.kernel, filter_params=True, **params)

    @property
    def _pairwise(self):
        return self.kernel == "precomputed"

In [5]:
# Train kelmor on a dataset
Regularization_coefficient = 2 # Try different values
# 'num_pivots' will not work when method=="full"
tracemalloc.start()
start_time = time.time()
kelmor = KELMRPC(C=Regularization_coefficient, method="nfull", num_pivots=10000, kernel="rbf")
kelmor.fit(Xtra, Ytra)
print('Training time: %.4f sec' % ((time.time() - start_time)))
current, peak = tracemalloc.get_traced_memory()
print(f"Current memory usage: {current / (1024**2):.2f} MB")
print(f"Peak memory usage: {peak / (1024**2):.2f} MB")
tracemalloc.stop()

Training time: 83.2230 sec
Current memory usage: 6.75 MB
Peak memory usage: 7634.65 MB


In [6]:
# Predicting samples
tracemalloc.start()
start_time = time.time()
prediction = kelmor.predict(Xtst)
print('Inference time: %.4f sec' % ((time.time() - start_time)))
current, peak = tracemalloc.get_traced_memory()
print(f"Current memory usage: {current / (1024**2):.2f} MB")
print(f"Peak memory usage: {peak / (1024**2):.2f} MB")
tracemalloc.stop()

Inference time: 1.6524 sec
Current memory usage: 2.04 MB
Peak memory usage: 2034.51 MB


In [7]:
threshold = 0.0
num_rows = len(Ytst)
num_columns = len(Ytst[0])
total_elements = num_rows * num_columns

Ypred = (prediction > threshold).astype(int)
Ypred = 2*Ypred -1
error_count = (Ypred != Ytst).sum()
error_rate = error_count/total_elements

In [8]:
print(error_rate)

0.028604110411041102
